# 深度学习作业 HW0

姓名：王旭登
学号：20234080426

---

# 2 循环神经网络

## 2.1 理论计算题

### 问题描述

给定一个字符序列 "ababc"，假设采用一阶马尔可夫模型（即 p(x_t | x_{t-1})），使用拉普拉斯平滑（加1平滑）估计以下条件概率：

1. p('a' | 'b')
2. p('c' | 'b')

词汇表为 {'a','b','c'}，计算时考虑所有可能转移，包括未出现的情况。

### 解答

#### 步骤1：统计字符转移次数

首先，我们需要从序列 "ababc" 中提取所有相邻字符对：

- a → b
- b → a
- a → b
- b → c

统计每个前缀字符后面跟随的字符出现次数：

- 前缀 'a' 后面出现：b, b → 转移次数：a→b=2
- 前缀 'b' 后面出现：a, c → 转移次数：b→a=1, b→c=1
- 前缀 'c' 后面出现：无（'c'在序列末尾）→ 转移次数：c→a=0, c→b=0, c→c=0

#### 步骤2：应用拉普拉斯平滑

拉普拉斯平滑的公式为：

$$p(x_t | x_{t-1}) = \frac{count(x_{t-1}, x_t) + 1}{count(x_{t-1}) + V}$$

其中：
- $count(x_{t-1}, x_t)$：从 $x_{t-1}$ 转移到 $x_t$ 的次数
- $count(x_{t-1})$：前缀 $x_{t-1}$ 出现的总次数
- $V$：词汇表大小（这里 V=3）

#### 步骤3：计算条件概率

**1. p('a' | 'b')**

- $count(b, a) = 1$
- $count(b) = 2$（'b'作为前缀出现2次）
- $V = 3$

$$p(a | b) = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$$

**2. p('c' | 'b')**

- $count(b, c) = 1$
- $count(b) = 2$
- $V = 3$

$$p(c | b) = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$$

#### 步骤4：验证所有转移概率和为1

对于前缀 'b'，所有可能的转移：

- $p(a | b) = \frac{2}{5} = 0.4$
- $p(b | b) = \frac{0 + 1}{2 + 3} = \frac{1}{5} = 0.2$
- $p(c | b) = \frac{2}{5} = 0.4$

总和：$0.4 + 0.2 + 0.4 = 1.0$，验证通过。

### 最终答案

1. $p('a' | 'b') = \frac{2}{5} = 0.4$
2. $p('c' | 'b') = \frac{2}{5} = 0.4$

## 2.2 编程题

### 问题描述

编写一个函数 `preprocess_text(text, n)`，完成以下步骤：

1. 将文本转换为小写，去除标点符号（保留字母和空格）。
2. 按空格分词。
3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）。
4. 用滑动窗口生成长度为n的特征序列和对应的下一个词标签（用于自回归语言模型）。

返回词汇表字典和 (特征列表, 标签列表)。例如，输入 "The time machine" 和 n=2，应生成特征 `[['the','time'], ['time','machine']]` 和标签 `['machine', None]`（若无后续词则忽略）。

### 代码实现

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    预处理文本并生成特征和标签
    
    参数:
        text: 输入文本
        n: 滑动窗口大小
    
    返回:
        vocab: 词汇表字典 {词: ID}
        (features, labels): 特征列表和标签列表
    """
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    tokens = text.split()
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(tokens)
    sorted_words = sorted(word_counts.keys(), key=lambda x: (-word_counts[x], x))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 4. 用滑动窗口生成长度为n的特征序列和对应的下一个词标签
    features = []
    labels = []
    
    for i in range(len(tokens) - n):
        feature = tokens[i:i+n]
        label = tokens[i+n] if (i+n) < len(tokens) else None
        features.append(feature)
        labels.append(label)
    
    return vocab, (features, labels)

# 测试示例
text = "The time machine"
vocab, (features, labels) = preprocess_text(text, n=2)

print("词汇表:", vocab)
print("特征:", features)
print("标签:", labels)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time']]
标签: ['machine']


# 3 循环神经网络

## 3.1 理论计算题

### 问题描述

考虑一个线性RNN（无偏置），定义为：

$$h_t = W_{hh} h_{t-1} + W_{hx} x_t$$

$$o_t = W_{oh} h_t$$

假设损失函数为平方损失：

$$L = \frac{1}{2} \sum_{t=1}^T (o_t - y_t)^2$$

推导损失对权重 $W_{hh}$ 的梯度表达式（通过时间反向传播，展开到所有时间步），并说明梯度消失或爆炸的条件。

### 解答

#### 步骤1：定义变量和损失函数

已知：
- 隐藏状态：$h_t = W_{hh} h_{t-1} + W_{hx} x_t$
- 输出：$o_t = W_{oh} h_t$
- 损失：$L = \frac{1}{2} \sum_{t=1}^T (o_t - y_t)^2$

#### 步骤2：计算损失对输出的梯度

$$\frac{\partial L}{\partial o_t} = o_t - y_t$$

#### 步骤3：计算损失对隐藏状态的梯度

损失对 $h_t$ 的梯度需要考虑当前时间步和后续时间步的影响：

$$\frac{\partial L}{\partial h_t} = \frac{\partial o_t}{\partial h_t} \cdot \frac{\partial L}{\partial o_t} + \frac{\partial h_{t+1}}{\partial h_t} \cdot \frac{\partial L}{\partial h_{t+1}}$$

其中：
- $\frac{\partial o_t}{\partial h_t} = W_{oh}^T$
- $\frac{\partial h_{t+1}}{\partial h_t} = W_{hh}^T$

因此：

$$\delta_t = \frac{\partial L}{\partial h_t} = W_{oh}^T (o_t - y_t) + W_{hh}^T \delta_{t+1}$$

边界条件：$\delta_{T+1} = 0$（最后一个时间步没有后续影响）

#### 步骤4：计算损失对 $W_{hh}$ 的梯度

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \frac{\partial L}{\partial h_t} \cdot \frac{\partial h_t}{\partial W_{hh}}$$

由于 $h_t = W_{hh} h_{t-1} + W_{hx} x_t$，所以：

$$\frac{\partial h_t}{\partial W_{hh}} = h_{t-1}^T$$

因此：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \delta_t h_{t-1}^T$$

将 $\delta_t$ 展开，得到递推关系：

$$\delta_t = W_{oh}^T (o_t - y_t) + W_{hh}^T \delta_{t+1}$$

展开到所有时间步：

$$\delta_t = \sum_{k=t}^T (W_{hh}^T)^{k-t} W_{oh}^T (o_k - y_k)$$

因此，梯度可以表示为：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \left( \sum_{k=t}^T (W_{hh}^T)^{k-t} W_{oh}^T (o_k - y_k) \right) h_{t-1}^T$$

#### 步骤5：梯度消失或爆炸的条件

从梯度表达式可以看出，梯度中包含 $(W_{hh}^T)^{k-t}$ 项，这是一个矩阵幂次。

**梯度消失条件**：
- 当 $W_{hh}$ 的所有特征值的绝对值都小于1时
- $(W_{hh}^T)^{k-t}$ 随着时间步差 $k-t$ 的增大而趋近于0
- 导致早期时间步的梯度贡献几乎为0

**梯度爆炸条件**：
- 当 $W_{hh}$ 的最大特征值的绝对值大于1时
- $(W_{hh}^T)^{k-t}$ 随着时间步差 $k-t$ 的增大而指数增长
- 导致梯度值变得非常大，训练不稳定

### 最终答案

损失对权重 $W_{hh}$ 的梯度表达式：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \delta_t h_{t-1}^T$$

其中 $\delta_t = W_{oh}^T (o_t - y_t) + W_{hh}^T \delta_{t+1}$，$\delta_{T+1} = 0$。

- **梯度消失**：当 $W_{hh}$ 的所有特征值绝对值 < 1
- **梯度爆炸**：当 $W_{hh}$ 的最大特征值绝对值 > 1

## 3.2 编程题

### 问题描述

实现一个简单的RNN单元的前向传播和单步反向传播（仅计算梯度，不更新）。给定输入 $x_t$（形状 (batch_size, input_size)）、上一隐藏状态 $h_{prev}$（形状 (batch_size, hidden_size)），以及权重 $W_{hx}, W_{hh}, b_h$，计算当前隐藏状态 $h_t$。同时实现反向传播，已知上游梯度 $dh_{next}$（即损失对 $h_t$ 的梯度），计算 $dx_t, dh_{prev}, dW_{hx}, dW_{hh}, db_h$（使用 tanh 激活函数）。

### 代码实现

In [2]:
import numpy as np

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单元前向传播
    
    参数:
        x_t: 当前时间步输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入到隐藏的权重，形状 (hidden_size, input_size)
        W_hh: 隐藏到隐藏的权重，形状 (hidden_size, hidden_size)
        b_h: 隐藏层偏置，形状 (hidden_size,)
    
    返回:
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 缓存用于反向传播的中间结果
    """
    # 计算隐藏状态的线性部分
    pre_activation = np.dot(x_t, W_hx.T) + np.dot(h_prev, W_hh.T) + b_h
    
    # 应用tanh激活函数
    h_t = np.tanh(pre_activation)
    
    # 缓存中间结果
    cache = (x_t, h_prev, pre_activation, h_t)
    
    return h_t, cache

def rnn_backward(dh_next, cache):
    """
    RNN单元单步反向传播
    
    参数:
        dh_next: 上游梯度（损失对h_t的梯度），形状 (batch_size, hidden_size)
        cache: 前向传播的缓存
    
    返回:
        dx_t: 输入梯度，形状 (batch_size, input_size)
        dh_prev: 上一隐藏状态梯度，形状 (batch_size, hidden_size)
        dW_hx: 输入到隐藏权重梯度，形状 (hidden_size, input_size)
        dW_hh: 隐藏到隐藏权重梯度，形状 (hidden_size, hidden_size)
        db_h: 偏置梯度，形状 (hidden_size,)
    """
    x_t, h_prev, pre_activation, h_t = cache
    
    batch_size = x_t.shape[0]
    
    # tanh的导数: (1 - tanh(x)^2)
    dtanh = 1 - h_t ** 2
    
    # 梯度通过激活函数
    d_pre_activation = dh_next * dtanh
    
    # 计算输入梯度
    dx_t = np.dot(d_pre_activation, W_hx)
    
    # 计算上一隐藏状态梯度
    dh_prev = np.dot(d_pre_activation, W_hh)
    
    # 计算权重梯度
    dW_hx = np.dot(d_pre_activation.T, x_t) / batch_size
    dW_hh = np.dot(d_pre_activation.T, h_prev) / batch_size
    
    # 计算偏置梯度
    db_h = np.mean(d_pre_activation, axis=0)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 测试示例
np.random.seed(42)

batch_size = 2
input_size = 3
hidden_size = 4

# 随机初始化参数
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(hidden_size, input_size)
W_hh = np.random.randn(hidden_size, hidden_size)
b_h = np.random.randn(hidden_size)

# 前向传播
h_t, cache = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)
print("前向传播结果 h_t 形状:", h_t.shape)

# 反向传播（模拟上游梯度）
dh_next = np.random.randn(batch_size, hidden_size)
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_backward(dh_next, cache)

print("反向传播结果:")
print("dx_t 形状:", dx_t.shape)
print("dh_prev 形状:", dh_prev.shape)
print("dW_hx 形状:", dW_hx.shape)
print("dW_hh 形状:", dW_hh.shape)
print("db_h 形状:", db_h.shape)

前向传播结果 h_t 形状: (2, 4)
反向传播结果:
dx_t 形状: (2, 3)
dh_prev 形状: (2, 4)
dW_hx 形状: (4, 3)
dW_hh 形状: (4, 4)
db_h 形状: (4,)


# 4 高级循环神经网络

## 4.1 理论计算题

### 问题描述

假设一个深度双向RNN，有L层，每层隐藏单元数为H，输入维度为D，输出维度为O（仅考虑最后输出层）。计算该模型的参数总数（包括所有全连接层的权重和偏置），忽略嵌入层和输出层之前的投影，明确给出表达式。

### 解答

#### 步骤1：分析深度双向RNN的结构

一个深度双向RNN包含：
- L层，每层有前向和后向两个方向
- 每层的前向和后向RNN单元
- 输出层（仅考虑最后一层的输出）

#### 步骤2：计算每层的参数

对于每层：

**前向RNN单元**：
- $W_{hx}^f$：输入到隐藏的权重，形状 (H, D) 或 (H, 2H)（非首层）
- $W_{hh}^f$：隐藏到隐藏的权重，形状 (H, H)
- $b_h^f$：偏置，形状 (H,)

**后向RNN单元**：
- $W_{hx}^b$：输入到隐藏的权重，形状 (H, D) 或 (H, 2H)（非首层）
- $W_{hh}^b$：隐藏到隐藏的权重，形状 (H, H)
- $b_h^b$：偏置，形状 (H,)

#### 步骤3：分层计算参数

**第一层（输入层）**：
- 输入维度：D
- 前向：$H \times D$（W_hx） + $H \times H$（W_hh） + $H$（b_h）
- 后向：$H \times D$（W_hx） + $H \times H$（W_hh） + $H$（b_h）
- 总计：$2 \times (H \times D + H \times H + H)$

**中间层和最后一层（共L-1层）**：
- 输入维度：2H（前向和后向拼接）
- 每层前向：$H \times 2H$（W_hx） + $H \times H$（W_hh） + $H$（b_h）
- 每层后向：$H \times 2H$（W_hx） + $H \times H$（W_hh） + $H$（b_h）
- 每层总计：$2 \times (H \times 2H + H \times H + H)$

**输出层**：
- 输入维度：2H（最后一层的前向和后向拼接）
- 权重：$O \times 2H$
- 偏置：$O$
- 总计：$O \times 2H + O$

#### 步骤4：总参数计算

$$\text{总参数} = \text{第一层参数} + (L-1) \times \text{中间层参数} + \text{输出层参数}$$

代入公式：

$$\text{总参数} = 2(H \cdot D + H^2 + H) + (L-1) \cdot 2(2H^2 + H^2 + H) + O(2H + 1)$$

化简：

$$\text{总参数} = 2H(D + H + 1) + 2(L-1)H(3H + 1) + O(2H + 1)$$

### 最终答案

深度双向RNN的参数总数为：

$$\boxed{2H(D + H + 1) + 2(L-1)H(3H + 1) + O(2H + 1)}$$

其中：
- L：层数
- H：每层隐藏单元数
- D：输入维度
- O：输出维度

## 4.2 编程题

### 问题描述

实现一个双向RNN编码器，接收序列X（形状 (seq_len, batch, input_dim)），使用torch.nn.RNN或手动实现。要求返回每个时间步的拼接后的前向和后向隐藏状态（形状 (seq_len, batch, 2*hidden_dim)），以及最终时间步的拼接隐藏状态（作为序列表示）。

### 代码实现

In [3]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    """
    双向RNN编码器
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BidirectionalRNNEncoder, self).__init__()

        self.hidden_dim = hidden_dim
        
        # 使用PyTorch内置的双向RNN
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False
        )

    def forward(self, X):
        """
        前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回:
            outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            final_hidden: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
        """
        # 前向传播
        # outputs: (seq_len, batch, 2*hidden_dim)
        # h_n: (2*num_layers, batch, hidden_dim)
        outputs, h_n = self.rnn(X)
        
        # 获取最终时间步的隐藏状态
        # 前向最后一层的最后时间步: h_n[-2]
        # 后向最后一层的最后时间步: h_n[-1]
        forward_final = h_n[-2]  # (batch, hidden_dim)
        backward_final = h_n[-1]  # (batch, hidden_dim)
        
        # 拼接前向和后向隐藏状态
        final_hidden = torch.cat([forward_final, backward_final], dim=1)  # (batch, 2*hidden_dim)
        
        return outputs, final_hidden

# 测试示例
torch.manual_seed(42)

seq_len = 5
batch_size = 3
input_dim = 10
hidden_dim = 20

# 随机生成输入
X = torch.randn(seq_len, batch_size, input_dim)

# 创建编码器
encoder = BidirectionalRNNEncoder(input_dim, hidden_dim)

# 前向传播
outputs, final_hidden = encoder(X)

print("输出形状:")
print("outputs:", outputs.shape)  # 期望: (seq_len, batch, 2*hidden_dim)
print("final_hidden:", final_hidden.shape)  # 期望: (batch, 2*hidden_dim)

输出形状:
outputs: torch.Size([5, 3, 40])
final_hidden: torch.Size([3, 40])


# 5 嵌入向量

## 5.1 理论计算题

### 问题描述

在Skip-gram模型中，给定中心词 $w_c$ 和上下文词 $w_o$，使用负采样（采样K个负样本）。推导其损失函数（对数似然）的表达式，并说明如何从噪声分布中采样负样本。假设词向量为 $v_c, u_o$，负样本词向量为 $u_{nk}$，写出完整的目标函数。

### 解答

#### 步骤1：Skip-gram模型的基本思想

Skip-gram模型的目标是给定中心词预测上下文词。在负采样中，我们最大化正样本对的概率，同时最小化负样本对的概率。

#### 步骤2：正样本和负样本

- **正样本**：$(w_c, w_o)$ - 真实的中心词-上下文词对
- **负样本**：$(w_c, w_{n1}), (w_c, w_{n2}), ..., (w_c, w_{nK})$ - 从噪声分布采样的假样本

#### 步骤3：概率计算

使用sigmoid函数计算概率：

$$P(D=1 | w_c, w_o) = \sigma(v_c^T u_o)$$
$$P(D=0 | w_c, w_o) = 1 - \sigma(v_c^T u_o) = \sigma(-v_c^T u_o)$$

其中 $\sigma(x) = \frac{1}{1 + e^{-x}}$ 是sigmoid函数。

#### 步骤4：损失函数推导

目标是最大化正样本的对数似然，同时最小化负样本的对数似然：

$$\mathcal{L} = \log P(D=1 | w_c, w_o) + \sum_{k=1}^K \log P(D=0 | w_c, w_{nk})$$

代入概率表达式：

$$\mathcal{L} = \log \sigma(v_c^T u_o) + \sum_{k=1}^K \log \sigma(-v_c^T u_{nk})$$

由于训练时我们最小化损失，所以通常取负号：

$$\mathcal{L} = -\left[ \log \sigma(v_c^T u_o) + \sum_{k=1}^K \log \sigma(-v_c^T u_{nk}) \right]$$

#### 步骤5：负样本采样方法

负样本通常从噪声分布 $P_n(w)$ 中采样，噪声分布通常设置为词频的3/4次方：

$$P_n(w) \propto f(w)^{3/4}$$

其中 $f(w)$ 是词 $w$ 在语料库中的出现频率。

这样做的原因：
- 高频词更可能被选为负样本
- 但通过取3/4次方降低了高频词的优势
- 平衡了高频词和低频词的采样概率

#### 步骤6：完整目标函数

综合以上，完整的目标函数为：

$$\mathcal{L} = -\left[ \log \sigma(v_c^T u_o) + \sum_{k=1}^K \log \sigma(-v_c^T u_{nk}) \right]$$

其中：
- $v_c$：中心词的输入向量
- $u_o$：正样本上下文词的输出向量
- $u_{nk}$：第k个负样本词的输出向量
- $K$：负样本数量

### 最终答案

**损失函数**：

$$\boxed{\mathcal{L} = -\left[ \log \sigma(v_c^T u_o) + \sum_{k=1}^K \log \sigma(-v_c^T u_{nk}) \right]}$$

**负样本采样**：从噪声分布 $P_n(w) \propto f(w)^{3/4}$ 中采样K个负样本词。

## 5.2 编程题

### 问题描述

实现CBOW模型的前向传播和损失计算（不使用负采样，仅用完整softmax）。给定一批上下文词的索引列表（每个样本有context_size个上下文词），词汇表大小V，嵌入维度d。输入权重矩阵W（形状 (V, d)）和输出权重矩阵W_out（形状 (d, V)）。计算平均上下文向量作为隐藏层，然后计算输出概率分布，并计算交叉熵损失（目标为中心词索引）。返回损失值。

### 代码实现

In [4]:
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, center_indices, W, W_out):
    """
    CBOW模型前向传播和损失计算
    
    参数:
        context_indices: 上下文词索引，形状 (batch_size, context_size)
        center_indices: 中心词索引，形状 (batch_size,)
        W: 输入嵌入矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
    
    返回:
        loss: 交叉熵损失值
        probs: 输出概率分布，形状 (batch_size, V)
    """
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    
    # 1. 获取上下文词的嵌入向量
    # context_embeddings: (batch_size, context_size, d)
    context_embeddings = W[context_indices]
    
    # 2. 计算平均上下文向量作为隐藏层
    # hidden: (batch_size, d)
    hidden = torch.mean(context_embeddings, dim=1)
    
    # 3. 计算输出层（logits）
    # logits: (batch_size, V)
    logits = hidden @ W_out
    
    # 4. 计算softmax概率
    probs = F.softmax(logits, dim=1)
    
    # 5. 计算交叉熵损失
    loss = F.cross_entropy(logits, center_indices)
    
    return loss, probs

# 测试示例
torch.manual_seed(42)

batch_size = 2
context_size = 4
V = 100  # 词汇表大小
d = 50   # 嵌入维度

# 随机初始化权重
W = torch.randn(V, d)
W_out = torch.randn(d, V)

# 随机生成上下文词索引和中心词索引
context_indices = torch.randint(0, V, (batch_size, context_size))
center_indices = torch.randint(0, V, (batch_size,))

print("输入形状:")
print("context_indices:", context_indices.shape)
print("center_indices:", center_indices.shape)

# 前向传播
loss, probs = cbow_forward(context_indices, center_indices, W, W_out)

print("\n输出:")
print("损失值:", loss.item())
print("概率分布形状:", probs.shape)
print("概率和:", probs.sum(dim=1))

输入形状:
context_indices: torch.Size([2, 4])
center_indices: torch.Size([2])

输出:
损失值: 10.045507431030273
概率分布形状: torch.Size([2, 100])
概率和: tensor([1.0000, 1.0000])


# 6 注意力机制

## 6.1 理论计算题

### 问题描述

给定查询矩阵 $Q \in \mathbb{R}^{2 \times 4}$，键矩阵 $K \in \mathbb{R}^{3 \times 4}$，值矩阵 $V \in \mathbb{R}^{3 \times 5}$。计算缩放点积注意力（无掩码）的输出矩阵，要求写出中间步骤（先计算得分矩阵，再softmax，再加权求和）。使用 $\text{score} = \frac{Q K^T}{\sqrt{d_k}}$（$d_k = 4$）。可以只列出数值计算过程（用符号或具体数值）。

### 解答

#### 步骤1：定义输入矩阵

假设输入矩阵为：

$$Q = \begin{bmatrix}
q_{11} & q_{12} & q_{13} & q_{14} \\
q_{21} & q_{22} & q_{23} & q_{24}
\end{bmatrix}_{2 \times 4}$$

$$K = \begin{bmatrix}
k_{11} & k_{12} & k_{13} & k_{14} \\
k_{21} & k_{22} & k_{23} & k_{24} \\
k_{31} & k_{32} & k_{33} & k_{34}
\end{bmatrix}_{3 \times 4}$$

$$V = \begin{bmatrix}
v_{11} & v_{12} & v_{13} & v_{14} & v_{15} \\
v_{21} & v_{22} & v_{23} & v_{24} & v_{25} \\
v_{31} & v_{32} & v_{33} & v_{34} & v_{35}
\end{bmatrix}_{3 \times 5}$$

#### 步骤2：计算得分矩阵

得分矩阵 $S = \frac{Q K^T}{\sqrt{d_k}}$，其中 $d_k = 4$，所以 $\sqrt{d_k} = 2$。

首先计算 $Q K^T$：

$$Q K^T = \begin{bmatrix}
\sum_{i=1}^4 q_{1i}k_{1i} & \sum_{i=1}^4 q_{1i}k_{2i} & \sum_{i=1}^4 q_{1i}k_{3i} \\
\sum_{i=1}^4 q_{2i}k_{1i} & \sum_{i=1}^4 q_{2i}k_{2i} & \sum_{i=1}^4 q_{2i}k_{3i}
\end{bmatrix}_{2 \times 3}$$

然后除以 $\sqrt{d_k} = 2$：

$$S = \frac{1}{2} \times Q K^T = \begin{bmatrix}
s_{11} & s_{12} & s_{13} \\
s_{21} & s_{22} & s_{23}
\end{bmatrix}_{2 \times 3}$$

其中 $s_{ij} = \frac{1}{2} \sum_{m=1}^4 q_{im}k_{jm}$。

#### 步骤3：对得分矩阵应用softmax

对得分矩阵的每一行应用softmax：

$$\text{softmax}(s_i) = \left[ \frac{e^{s_{i1}}}{\sum_{j=1}^3 e^{s_{ij}}}, \frac{e^{s_{i2}}}{\sum_{j=1}^3 e^{s_{ij}}}, \frac{e^{s_{i3}}}{\sum_{j=1}^3 e^{s_{ij}}} \right]$$

得到注意力权重矩阵 $A$：

$$A = \begin{bmatrix}
a_{11} & a_{12} & a_{13} \\
a_{21} & a_{22} & a_{23}
\end{bmatrix}_{2 \times 3}$$

其中 $a_{ij} = \frac{e^{s_{ij}}}{\sum_{k=1}^3 e^{s_{ik}}}$。

#### 步骤4：加权求和计算输出

输出矩阵 $O = A V$：

$$O = \begin{bmatrix}
a_{11}v_{11} + a_{12}v_{21} + a_{13}v_{31} & a_{11}v_{12} + a_{12}v_{22} + a_{13}v_{32} & \dots & a_{11}v_{15} + a_{12}v_{25} + a_{13}v_{35} \\
a_{21}v_{11} + a_{22}v_{21} + a_{23}v_{31} & a_{21}v_{12} + a_{22}v_{22} + a_{23}v_{32} & \dots & a_{21}v_{15} + a_{22}v_{25} + a_{23}v_{35}
\end{bmatrix}_{2 \times 5}$$

#### 步骤5：具体数值示例

假设具体数值：

$$Q = \begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0
\end{bmatrix}, \quad K = \begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0
\end{bmatrix}, \quad V = \begin{bmatrix}
1 & 2 & 3 & 4 & 5 \\
6 & 7 & 8 & 9 & 10 \\
11 & 12 & 13 & 14 & 15
\end{bmatrix}$$

计算过程：

1. $Q K^T = \begin{bmatrix}1 & 0 & 0 \\ 0 & 1 & 0\end{bmatrix}$
2. $S = \frac{1}{2} Q K^T = \begin{bmatrix}0.5 & 0 & 0 \\ 0 & 0.5 & 0\end{bmatrix}$
3. $A = \text{softmax}(S) = \begin{bmatrix}0.5498 & 0.2251 & 0.2251 \\ 0.2251 & 0.5498 & 0.2251\end{bmatrix}$
4. $O = A V \approx \begin{bmatrix}2.99 & 3.99 & 4.99 & 5.99 & 6.99 \\ 5.01 & 6.01 & 7.01 & 8.01 & 9.01\end{bmatrix}$

### 最终答案

缩放点积注意力的计算步骤：

1. **得分矩阵**：$S = \frac{Q K^T}{\sqrt{d_k}}$
2. **注意力权重**：$A = \text{softmax}(S)$
3. **输出**：$O = A V$

输出矩阵形状为 $2 \times 5$。

## 6.2 编程题

### 问题描述

实现多头注意力（Multi-Head Attention）的前向传播，假设num_heads=2，d_model=4。给定输入X（形状 (seq_len, batch, d_model)），分别线性投影得到Q, K, V（每个头的维度 $d_k = d_v = d_{model}/num_{heads}$）。对每个头计算缩放点积注意力，然后将所有头的输出拼接并经过最终线性层。返回输出（形状与输入相同）。

### 代码实现

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """
    多头注意力机制
    """
    def __init__(self, d_model=4, num_heads=2):
        super(MultiHeadAttention, self).__init__()

        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads

        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # 最终线性层
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V):
        """
        缩放点积注意力
        
        参数:
            Q: 查询矩阵，形状 (batch, num_heads, seq_len, d_k)
            K: 键矩阵，形状 (batch, num_heads, seq_len, d_k)
            V: 值矩阵，形状 (batch, num_heads, seq_len, d_v)
        
        返回:
            output: 注意力输出
            attn_weights: 注意力权重
        """
        # 计算得分矩阵
        # scores: (batch, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))

        # 应用softmax
        attn_weights = F.softmax(scores, dim=-1)

        # 加权求和
        output = torch.matmul(attn_weights, V)

        return output, attn_weights

    def forward(self, X):
        """
        前向传播
        
        参数:
            X: 输入，形状 (seq_len, batch, d_model)
        
        返回:
            output: 输出，形状 (seq_len, batch, d_model)
        """
        seq_len, batch_size, d_model = X.shape

        # 转换形状为 (batch, seq_len, d_model)
        X = X.permute(1, 0, 2)  # (batch, seq_len, d_model)

        # 线性投影得到Q, K, V
        Q = self.W_q(X)  # (batch, seq_len, d_model)
        K = self.W_k(X)  # (batch, seq_len, d_model)
        V = self.W_v(X)  # (batch, seq_len, d_model)

        # 重塑为多头格式
        # (batch, seq_len, d_model) -> (batch, num_heads, seq_len, d_k)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_v).transpose(1, 2)

        # 计算缩放点积注意力
        attn_output, attn_weights = self.scaled_dot_product_attention(Q, K, V)
        # attn_output: (batch, num_heads, seq_len, d_v)

        # 拼接所有头的输出
        # (batch, num_heads, seq_len, d_v) -> (batch, seq_len, num_heads * d_v) = (batch, seq_len, d_model)
        concat_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        # 最终线性层
        output = self.W_o(concat_output)  # (batch, seq_len, d_model)

        # 转换回原始形状 (seq_len, batch, d_model)
        output = output.permute(1, 0, 2)

        return output, attn_weights

# 测试示例
torch.manual_seed(42)

seq_len = 3
batch_size = 2
d_model = 4
num_heads = 2

# 随机生成输入
X = torch.randn(seq_len, batch_size, d_model)

# 创建多头注意力层
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

# 前向传播
output, attn_weights = mha(X)

print("输入形状:", X.shape)
print("输出形状:", output.shape)
print("注意力权重形状:", attn_weights.shape)

# 验证输出形状与输入相同
assert output.shape == X.shape, "输出形状应与输入相同"

输入形状: torch.Size([3, 2, 4])
输出形状: torch.Size([3, 2, 4])
注意力权重形状: torch.Size([2, 2, 3, 3])
